<a href="https://colab.research.google.com/github/Meru-dog/credit-default-threshold/blob/main/credit_default_wandb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q wandb lightgbm ucimlrepo
!pip install -q wandb lightgbm ucimlrepo gradio

In [ ]:
import os
import wandb
from google.colab import userdata

PROJECT_NAME = "credit-default-ml"
RANDOM_STATE = 42
NOTEBOOK_VERSION = "v1"

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

with wandb.init(
    project=PROJECT_NAME,
    name="setup-check",
    job_type="setup",
    config={
        "random_state": RANDOM_STATE,
        "notebook_version": NOTEBOOK_VERSION,
    },
) as run:
    run.log({"setup_check": 1})

In [ ]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=350)

features_raw = dataset.data.features.copy()
target_raw = dataset.data.targets.copy()

display(dataset.variables[["name", "role", "description"]])

COLUMN_NAMES = (
    ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
    + ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    + [f"BILL_AMT{i}" for i in range(1, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 7)]
)
TARGET = "default"

assert features_raw.shape[1] == len(COLUMN_NAMES)
assert target_raw.shape[1] == 1

df = features_raw.copy()
df.columns = COLUMN_NAMES
df[TARGET] = target_raw.iloc[:, 0].astype(int)

print("Shape:", df.shape)
df.head()

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "min": df.min(numeric_only=True),
    "max": df.max(numeric_only=True),
})
display(summary)

print("Duplicate rows:", df.duplicated().sum())
print("\nTarget counts")
print(df[TARGET].value_counts())
print("\nTarget share (%)")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

In [ ]:
PAY_COLUMNS = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
BILL_COLUMNS = [f"BILL_AMT{i}" for i in range(1, 7)]
PAY_AMT_COLUMNS = [f"PAY_AMT{i}" for i in range(1, 7)]

display(pd.DataFrame({
    column: df[column].value_counts()
    for column in PAY_COLUMNS
}).fillna(0).astype(int).sort_index())

print("Rows with a negative bill amount:", (df[BILL_COLUMNS] < 0).any(axis=1).sum())
print("Negative payment values:", (df[PAY_AMT_COLUMNS] < 0).sum().sum())
print("Rows with LIMIT_BAL <= 0:", (df["LIMIT_BAL"] <= 0).sum())

In [ ]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
FEATURES = ["LIMIT_BAL"] + PAY_COLUMNS + BILL_COLUMNS + PAY_AMT_COLUMNS
EXCLUDED = ["SEX", "MARRIAGE", "AGE", "EDUCATION"]

assert len(FEATURES) == 19
assert not set(EXCLUDED) & set(FEATURES)
assert df[FEATURES + [TARGET]].isna().sum().sum() == 0

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

In [ ]:
assert not set(X_train.index) & set(X_test.index)

print("Train:", X_train.shape, f"default rate: {y_train.mean():.2%}")
print("Test:", X_test.shape, f"default rate: {y_test.mean():.2%}")

print("\nPAY codes in the training set")
for column in PAY_COLUMNS:
    print(column, sorted(X_train[column].unique().tolist()))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_scaled_pipeline(model):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", model),
    ])

def make_unscaled_pipeline(model):
    return Pipeline([
        ("model", model),
    ])

In [ ]:
import numpy as np

X_train_scaled = StandardScaler().fit_transform(X_train)

print("Shape:", X_train_scaled.shape)
print("Missing values:", np.isnan(X_train_scaled).sum())
print("Largest absolute mean:", np.abs(X_train_scaled.mean(axis=0)).max().round(6))
print("Standard deviation range:", X_train_scaled.std(axis=0).min().round(6), "to", X_train_scaled.std(axis=0).max().round(6))

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

CV_FOLDS = 5
SCORING = {
    "pr_auc": "average_precision",
    "brier": "neg_brier_score",
    "roc_auc": "roc_auc",
}

cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
results = []

def run_cv_experiment(run_name, pipeline):
    model = pipeline.named_steps["model"]

    config = {
        "model_name": type(model).__name__,
        "model_params": model.get_params(),
        "dataset": "uci-350",
        "features": FEATURES,
        "test_size": TEST_SIZE,
        "train_target_rate": float(y_train.mean()),
        "cv_folds": CV_FOLDS,
        "random_state": RANDOM_STATE,
        "notebook_version": NOTEBOOK_VERSION,
    }

    with wandb.init(
        project=PROJECT_NAME,
        name=run_name,
        job_type="cross-validation",
        config=config,
    ) as run:
        scores = cross_validate(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring=SCORING,
        )

        summary = {"run_name": run_name}
        for metric in SCORING:
            values = scores[f"test_{metric}"]
            if metric == "brier":
                values = -values
            summary[f"cv_{metric}_mean"] = float(values.mean())
            summary[f"cv_{metric}_std"] = float(values.std())

        run.summary.update(summary)

    results.append(summary)
    return summary

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

run_cv_experiment(
    "baseline-majority",
    make_unscaled_pipeline(DummyClassifier(strategy="prior")),
)

run_cv_experiment(
    "baseline-logistic-regression",
    make_scaled_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    ),
)

pd.DataFrame(results).round(4)

In [ ]:
from lightgbm import LGBMClassifier

lgbm_settings = [
    {
        "run_name": "lgbm-small",
        "num_leaves": 7,
        "n_estimators": 200,
    },
    {
        "run_name": "lgbm-medium",
        "num_leaves": 15,
        "n_estimators": 300,
    },
    {
        "run_name": "lgbm-large",
        "num_leaves": 31,
        "n_estimators": 400,
    },
]

for setting in lgbm_settings:
    model = LGBMClassifier(
        num_leaves=setting["num_leaves"],
        n_estimators=setting["n_estimators"],
        learning_rate=0.05,
        min_child_samples=50,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    run_cv_experiment(setting["run_name"], make_unscaled_pipeline(model))

In [ ]:
results_df = (
    pd.DataFrame(results)
    .drop_duplicates("run_name", keep="last")
    .sort_values("cv_pr_auc_mean", ascending=False)
)

results_df[[
    "run_name",
    "cv_pr_auc_mean",
    "cv_pr_auc_std",
    "cv_brier_mean",
    "cv_brier_std",
    "cv_roc_auc_mean",
    "cv_roc_auc_std",
]].round(4)

In [ ]:
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

SELECTED_PARAMS = {
    "num_leaves": 7,
    "n_estimators": 200,
    "learning_rate": 0.05,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}

def make_selected_model():
    return make_unscaled_pipeline(LGBMClassifier(**SELECTED_PARAMS))

def make_calibrated_model(method):
    return CalibratedClassifierCV(
        estimator=make_selected_model(),
        method=method,
        cv=3,
    )

calibration_options = {
    "uncalibrated": make_selected_model(),
    "sigmoid": make_calibrated_model("sigmoid"),
    "isotonic": make_calibrated_model("isotonic"),
}

def evaluate_out_of_fold(model):
    probabilities = np.zeros(len(y_train))
    fold_scores = []

    for train_index, valid_index in cv.split(X_train, y_train):
        fold_model = clone(model)
        fold_model.fit(X_train.iloc[train_index], y_train.iloc[train_index])

        fold_probabilities = fold_model.predict_proba(X_train.iloc[valid_index])[:, 1]
        probabilities[valid_index] = fold_probabilities
        fold_y = y_train.iloc[valid_index]

        fold_scores.append({
            "pr_auc": average_precision_score(fold_y, fold_probabilities),
            "brier": brier_score_loss(fold_y, fold_probabilities),
            "roc_auc": roc_auc_score(fold_y, fold_probabilities),
        })

    return probabilities, pd.DataFrame(fold_scores)

oof_probabilities = {}
calibration_rows = []

for name, model in calibration_options.items():
    probabilities, fold_scores = evaluate_out_of_fold(model)
    oof_probabilities[name] = probabilities

    row = {"option": name}
    for metric in fold_scores.columns:
        row[f"{metric}_mean"] = fold_scores[metric].mean()
        row[f"{metric}_std"] = fold_scores[metric].std(ddof=0)
    calibration_rows.append(row)

calibration_df = pd.DataFrame(calibration_rows).set_index("option")
calibration_df.round(4)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")

for name, probabilities in oof_probabilities.items():
    observed, predicted = calibration_curve(
        y_train,
        probabilities,
        n_bins=10,
        strategy="quantile",
    )
    ax.plot(predicted, observed, marker="o", label=name)

ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed default rate")
ax.set_title("Calibration curves (training-set out-of-fold)")
ax.legend()
plt.show()

with wandb.init(
    project=PROJECT_NAME,
    name="calibration-comparison",
    job_type="calibration",
    config={
        "selected_model": "lgbm-small",
        "model_params": SELECTED_PARAMS,
        "calibration_cv_folds": 3,
        "outer_cv_folds": CV_FOLDS,
        "random_state": RANDOM_STATE,
        "notebook_version": NOTEBOOK_VERSION,
    },
) as run:
    run.log({
        "calibration_curves": wandb.Image(fig),
        "calibration_results": wandb.Table(
            dataframe=calibration_df.reset_index()
        ),
    })

In [ ]:
COST_RATIOS = {"low": 3, "base": 5, "high": 10}
THRESHOLD_GRID = np.round(np.arange(0.01, 1.00, 0.01), 2)

def evaluate_threshold(y_true, probabilities, threshold, cost_ratio):
    y_true = np.asarray(y_true)
    predicted = probabilities >= threshold

    tp = np.sum(predicted & (y_true == 1))
    fp = np.sum(predicted & (y_true == 0))
    fn = np.sum(~predicted & (y_true == 1))
    tn = np.sum(~predicted & (y_true == 0))

    return {
        "cost_per_customer": (cost_ratio * fn + fp) / len(y_true),
        "precision": tp / (tp + fp) if (tp + fp) else 0.0,
        "recall": tp / (tp + fn) if (tp + fn) else 0.0,
        "review_rate": predicted.mean(),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

def find_best_threshold(y_true, probabilities, cost_ratio):
    costs = [
        evaluate_threshold(y_true, probabilities, threshold, cost_ratio)["cost_per_customer"]
        for threshold in THRESHOLD_GRID
    ]
    return float(THRESHOLD_GRID[int(np.argmin(costs))])

selected_probabilities = oof_probabilities["uncalibrated"]
threshold_rows = []
selected_thresholds = {}

for scenario, cost_ratio in COST_RATIOS.items():
    best_threshold = find_best_threshold(y_train, selected_probabilities, cost_ratio)
    selected_thresholds[scenario] = best_threshold

    candidates = {
        "default": 0.5,
        "theoretical": 1 / (1 + cost_ratio),
        "cost_minimizing": best_threshold,
    }

    for threshold_type, threshold in candidates.items():
        threshold_rows.append({
            "scenario": scenario,
            "cost_ratio": cost_ratio,
            "threshold_type": threshold_type,
            "threshold": threshold,
            **evaluate_threshold(y_train, selected_probabilities, threshold, cost_ratio),
        })

    threshold_rows.append({
        "scenario": scenario,
        "cost_ratio": cost_ratio,
        "threshold_type": "review_nobody",
        "threshold": np.nan,
        "cost_per_customer": cost_ratio * y_train.mean(),
        "review_rate": 0.0,
    })
    threshold_rows.append({
        "scenario": scenario,
        "cost_ratio": cost_ratio,
        "threshold_type": "review_everyone",
        "threshold": np.nan,
        "cost_per_customer": 1 - y_train.mean(),
        "review_rate": 1.0,
    })

threshold_df = pd.DataFrame(threshold_rows)
PRIMARY_THRESHOLD = selected_thresholds["base"]

fold_thresholds = []
for _, valid_index in cv.split(X_train, y_train):
    fold_thresholds.append(find_best_threshold(
        y_train.iloc[valid_index],
        selected_probabilities[valid_index],
        COST_RATIOS["base"],
    ))

print("Primary threshold:", PRIMARY_THRESHOLD)
print("Best base-case threshold by fold:", fold_thresholds)

threshold_df[[
    "scenario",
    "threshold_type",
    "threshold",
    "cost_per_customer",
    "precision",
    "recall",
    "review_rate",
]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for scenario, cost_ratio in COST_RATIOS.items():
    costs = [
        evaluate_threshold(y_train, selected_probabilities, threshold, cost_ratio)["cost_per_customer"]
        for threshold in THRESHOLD_GRID
    ]
    ax.plot(THRESHOLD_GRID, costs, label=f"{scenario} ({cost_ratio}:1)")
    ax.axvline(selected_thresholds[scenario], linestyle=":", color="gray")

ax.set_xlabel("Decision threshold")
ax.set_ylabel("Cost per customer")
ax.set_title("Cost curves (training-set out-of-fold)")
ax.legend()
plt.show()

with wandb.init(
    project=PROJECT_NAME,
    name="threshold-design",
    job_type="threshold-design",
    config={
        "selected_model": "lgbm-small",
        "calibration": "none",
        "cost_ratios": COST_RATIOS,
        "threshold_grid_step": 0.01,
        "random_state": RANDOM_STATE,
        "notebook_version": NOTEBOOK_VERSION,
    },
) as run:
    run.summary.update({
        "primary_threshold": PRIMARY_THRESHOLD,
        "selected_thresholds": selected_thresholds,
        "base_fold_thresholds": fold_thresholds,
    })
    run.log({
        "cost_curves": wandb.Image(fig),
        "threshold_results": wandb.Table(dataframe=threshold_df),
    })

In [ ]:
final_pipeline = make_selected_model()
final_pipeline.fit(X_train, y_train)

test_probabilities = final_pipeline.predict_proba(X_test)[:, 1]

test_metrics = {
    "test_pr_auc": average_precision_score(y_test, test_probabilities),
    "test_brier": brier_score_loss(y_test, test_probabilities),
    "test_roc_auc": roc_auc_score(y_test, test_probabilities),
}

test_rows = []

for scenario, cost_ratio in COST_RATIOS.items():
    candidates = {
        "default": 0.5,
        "theoretical": 1 / (1 + cost_ratio),
        "cost_minimizing": selected_thresholds[scenario],
    }

    for threshold_type, threshold in candidates.items():
        test_rows.append({
            "scenario": scenario,
            "threshold_type": threshold_type,
            "threshold": threshold,
            **evaluate_threshold(y_test, test_probabilities, threshold, cost_ratio),
        })

    test_rows.append({
        "scenario": scenario,
        "threshold_type": "review_nobody",
        "threshold": np.nan,
        "cost_per_customer": cost_ratio * y_test.mean(),
        "review_rate": 0.0,
    })
    test_rows.append({
        "scenario": scenario,
        "threshold_type": "review_everyone",
        "threshold": np.nan,
        "cost_per_customer": 1 - y_test.mean(),
        "review_rate": 1.0,
    })

test_results_df = pd.DataFrame(test_rows)
test_predictions = (test_probabilities >= PRIMARY_THRESHOLD).astype(int)

with wandb.init(
    project=PROJECT_NAME,
    name="final-lgbm-small",
    job_type="final-evaluation",
    config={
        "selected_model": "lgbm-small",
        "model_params": SELECTED_PARAMS,
        "calibration": "none",
        "primary_threshold": PRIMARY_THRESHOLD,
        "selected_thresholds": selected_thresholds,
        "cost_ratios": COST_RATIOS,
        "dataset": "uci-350",
        "features": FEATURES,
        "random_state": RANDOM_STATE,
        "notebook_version": NOTEBOOK_VERSION,
    },
) as run:
    run.summary.update(test_metrics)
    run.log({
        "test_threshold_results": wandb.Table(dataframe=test_results_df),
        "test_confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_test.tolist(),
            preds=test_predictions.tolist(),
            class_names=["no_default", "default"],
        ),
    })

display(pd.Series(test_metrics).round(4).to_frame("score"))
display(test_results_df[[
    "scenario",
    "threshold_type",
    "threshold",
    "cost_per_customer",
    "precision",
    "recall",
    "review_rate",
    "tp",
    "fp",
    "fn",
    "tn",
]].round(4))

In [ ]:
import joblib
import lightgbm
import sklearn

MODEL_PATH = "credit_default_model.joblib"

model_bundle = {
    "pipeline": final_pipeline,
    "primary_threshold": PRIMARY_THRESHOLD,
    "selected_thresholds": selected_thresholds,
    "cost_ratios": COST_RATIOS,
    "features": FEATURES,
}
joblib.dump(model_bundle, MODEL_PATH)

with wandb.init(
    project=PROJECT_NAME,
    name="save-final-model",
    job_type="model-registration",
) as run:
    artifact = wandb.Artifact(
        name="credit-default-lgbm-small",
        type="model",
        description="LightGBM credit default model with cost-based threshold",
        metadata={
            "model_name": "LGBMClassifier",
            "model_params": SELECTED_PARAMS,
            "calibration": "none",
            "primary_threshold": PRIMARY_THRESHOLD,
            "selected_thresholds": selected_thresholds,
            "cost_ratios": COST_RATIOS,
            "features": FEATURES,
            "dataset": "uci-350",
            "sklearn_version": sklearn.__version__,
            "lightgbm_version": lightgbm.__version__,
            **{name: float(value) for name, value in test_metrics.items()},
        },
    )
    artifact.add_file(MODEL_PATH)
    run.log_artifact(artifact)

In [ ]:
loaded_bundle = joblib.load(MODEL_PATH)
loaded_probabilities = loaded_bundle["pipeline"].predict_proba(X_test)[:, 1]
loaded_predictions = (
    loaded_probabilities >= loaded_bundle["primary_threshold"]
).astype(int)

assert np.allclose(loaded_probabilities, test_probabilities)
assert np.array_equal(loaded_predictions, test_predictions)
print("Saved model verified")
print("Primary threshold:", loaded_bundle["primary_threshold"])

In [ ]:
import gradio as gr

simulator_bundle = joblib.load(MODEL_PATH)
simulator_probabilities = simulator_bundle["pipeline"].predict_proba(X_test)[:, 1]
simulator_default_rate = y_test.mean()

def simulate_threshold(cost_ratio, threshold):
    result = evaluate_threshold(
        y_test,
        simulator_probabilities,
        threshold,
        cost_ratio,
    )

    summary_df = pd.DataFrame([
        {"Metric": "Cost per customer", "Value": f"{result['cost_per_customer']:.4f}"},
        {"Metric": "Review rate", "Value": f"{result['review_rate']:.1%}"},
        {"Metric": "Precision", "Value": f"{result['precision']:.1%}"},
        {"Metric": "Recall", "Value": f"{result['recall']:.1%}"},
        {"Metric": "Theoretical threshold", "Value": f"{1 / (1 + cost_ratio):.3f}"},
        {"Metric": "Cost: review nobody", "Value": f"{cost_ratio * simulator_default_rate:.4f}"},
        {"Metric": "Cost: review everyone", "Value": f"{1 - simulator_default_rate:.4f}"},
    ])

    confusion_df = pd.DataFrame(
        [
            ["Actual: no default", result["tn"], result["fp"]],
            ["Actual: default", result["fn"], result["tp"]],
        ],
        columns=["", "Not reviewed", "Reviewed"],
    )

    costs = [
        evaluate_threshold(
            y_test,
            simulator_probabilities,
            grid_threshold,
            cost_ratio,
        )["cost_per_customer"]
        for grid_threshold in THRESHOLD_GRID
    ]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(THRESHOLD_GRID, costs)
    ax.axvline(threshold, color="tab:red", linestyle="--", label="Selected in simulator")
    ax.axvline(
        simulator_bundle["primary_threshold"],
        color="gray",
        linestyle=":",
        label="Project threshold (0.17)",
    )
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Cost per customer")
    ax.set_title(f"Test-set cost curve (cost ratio {cost_ratio}:1)")
    ax.legend()
    fig.tight_layout()
    plt.close(fig)

    return summary_df, confusion_df, fig

demo = gr.Interface(
    fn=simulate_threshold,
    inputs=[
        gr.Slider(1, 20, value=5, step=1, label="Missed default : false alert cost ratio"),
        gr.Slider(
            0.01,
            0.99,
            value=simulator_bundle["primary_threshold"],
            step=0.01,
            label="Decision threshold",
        ),
    ],
    outputs=[
        gr.Dataframe(label="Decision results"),
        gr.Dataframe(label="Confusion matrix"),
        gr.Plot(label="Cost curve"),
    ],
    live=True,
    title="Credit Default Threshold Simulator",
    description=(
        "Explore decision trade-offs on the test set using a fixed model. "
        "Cost ratios are assumptions. For learning only, not for real credit decisions."
    ),
    flagging_mode="never",
)

demo.launch(share=False)